In [ ]:
#| default_exp handlers.pipeline.writer

# Writer

Compatibility facade for `write_netcdf(tfm, cfg)` during the output modularization.

In [ ]:
#| export
from __future__ import annotations
from marisco.handlers.pipeline.output import (
    _SimpleBboxCB,
    build_global_attrs,
    project_netcdf_columns,
    validate_required_columns,
    write_netcdf,
)


## write_netcdf

Two-phase Strict guard → GlobAttrsFeeder → NetCDFEncoder:

| Phase | What | Fails with |
|-------|------|-----------|
| Guard 1 | `_MARIS_REQUIRED` ⊆ columns | `KeyError` |
| Guard 2 | `RenameColumnsCB` drops non-NC noise columns | silent |
| GlobAttrs | `BboxCB`, `DepthRangeCB`, `TimeRangeCB`, keywords, logs | `KeyError` on unknown attrs |
| Encode | `NetCDFEncoder.encode()` writes `.nc` | framework error |

In [ ]:
#| export
from marisco.handlers.pipeline.output import (
    _SimpleBboxCB,
    build_global_attrs,
    project_netcdf_columns,
    validate_required_columns,
    write_netcdf,
)


In [ ]:
from fastcore.test import test_eq
import pandas as pd
from marisco.callbacks import Transformer, RenameColumnsCB, get_time_units
from marisco.configs import NC_VARS
from marisco.handlers.pipeline.loader import _MARIS_REQUIRED
from netCDF4 import date2num

_t    = pd.Timestamp("2020-08-27", tz="UTC")
t_enc = date2num(_t.to_pydatetime(), units=get_time_units())
_df   = pd.DataFrame({
    "LAT":[79.5], "LON":[4.2], "TIME":[t_enc],
    "NUCLIDE":[28], "VALUE":[1.23e6], "UNC":[0.05e6], "UNIT":[9],
    "LAB":[281], "AREA":[2356], "SMP_DEPTH":[10.0], "SMP_ID":[1],
    "NOISE_COL":["drop_me"],    # NOT in NC_VARS -> dropped by Strict guard
    "PROVIDER_JUNK":["junk"],   # NOT in NC_VARS -> dropped by Strict guard
    "STATION":["FRAM_001"],     # IS in NC_VARS  -> kept
})

# Test 1: Guard drops non-NC_VARS noise, keeps valid NC_VARS cols
tfm_mock = Transformer({"SEAWATER": _df.copy()}, cbs=[], inplace=True); tfm_mock()
common_cols = set.intersection(*(set(df.columns) for df in tfm_mock.dfs.values()))
guard_rules = {k: k for k in NC_VARS if k in common_cols}
Transformer(tfm_mock.dfs, cbs=[RenameColumnsCB(guard_rules)], inplace=True)()
retained = sorted(tfm_mock.dfs["SEAWATER"].columns)
assert "NOISE_COL"     not in retained
assert "PROVIDER_JUNK" not in retained
assert "STATION" in retained and "LAT" in retained and "TIME" in retained
print(f"Guard Test 1 ✓ — retained: {retained}")
print(f"               dropped:  {sorted(set(_df.columns) - set(retained))}")

# Test 2: Phase 1 raises KeyError on missing required column
try:
    for grp, df in {"SEAWATER": _df.drop(columns=["LAT"]).copy()}.items():
        missing = _MARIS_REQUIRED - set(df.columns)
        if missing: raise KeyError(f"Group '{grp}' missing: {sorted(missing)}")
    assert False
except KeyError as e:
    print(f"Guard Test 2 ✓ — KeyError: {e}")

from marisco.handlers.pipeline.writer import write_netcdf
print("Guard Test 3 ✓ — write_netcdf importable")
print("\nAll write_netcdf guard smoke tests ✓")